In [ ]:
"""
Py-Microgrid Hybrid System Simulation Example
-----------------------------------
This example demonstrates how to:
1. Set up a hybrid system simulation
2. Download solar and wind resource data
3. Configure system parameters
4. Run optimization (single location or multiple locations)
5. Analyze and save results

Required files:
- Base YAML configuration file
- CSV file containing location data
"""

import os
import pandas as pd
from typing import Dict, List, Any

# Set NREL API key FIRST before any other imports to avoid timing issues
from py_microgrid.utilities.keys import set_developer_nrel_gov_key
set_developer_nrel_gov_key('ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3')

# Required imports (after API key is set)
from py_microgrid.utilities import ConfigManager
from py_microgrid.tools.optimization.system_optimizer import SystemOptimizer  
from py_microgrid.tools.optimization import LoadAnalyzer
from py_microgrid.tools.analysis.bos import EconomicCalculator
from py_microgrid.simulation.resource_files import ResourceDataManager

class HybridOptimizer:
    """Wrapper class for hybrid system optimization."""
    
    def __init__(self, 
                 yaml_file_path: str,
                 api_key: str,
                 email: str,
                 project_lifetime: int = 25,
                 discount_rate: float = 0.0588,
                 enable_flexible_load: bool = False,
                 max_load_reduction_percentage: float = 0.2):
        """
        Initialize hybrid system optimizer.
        
        Args:
            yaml_file_path: Path to YAML configuration file
            api_key: NREL API key for resource data
            email: Email for API authentication
            project_lifetime: Project lifetime in years
            discount_rate: Discount rate for economic calculations
            enable_flexible_load: Whether to enable flexible load management
            max_load_reduction_percentage: Maximum load reduction (if flexible load enabled)
        """
        self.yaml_file_path = yaml_file_path
        self.api_key = api_key
        self.email = email
        
        # Set up components
        self.resource_manager = ResourceDataManager(api_key, email)
        self.economic_calculator = EconomicCalculator(discount_rate, project_lifetime)
        self.system_optimizer = SystemOptimizer(
            yaml_file_path, 
            self.economic_calculator,
            enable_flexible_load=enable_flexible_load,
            max_load_reduction_percentage=max_load_reduction_percentage
        )

    def process_location(self, latitude: float, longitude: float, location_id: str = "") -> Dict[str, Any]:
        """Process single location optimization using Nelder-Mead."""
        print(f"Processing location {location_id} at ({latitude}, {longitude})")
        
        try:
            # Download solar and wind resource data
            solar_path = self.resource_manager.download_solar_data(
                latitude, longitude, "2022"  
            )
            wind_path = self.resource_manager.download_wind_data(
                latitude, longitude, "20220101", "20221231"  
            )
            
            # Update configuration with location and resource data
            config = self.system_optimizer.config_manager.load_yaml_safely(self.yaml_file_path)
            config['site']['data']['lat'] = latitude
            config['site']['data']['lon'] = longitude
            config['site']['solar_resource_file'] = solar_path.replace('\\', '/')
            config['site']['wind_resource_file'] = wind_path.replace('\\', '/')
            self.system_optimizer.config_manager.save_yaml_safely(config, self.yaml_file_path)
            
            # Check if grid is enabled to determine bounds
            grid_enabled = config.get('technologies', {}).get('grid', {}).get('enabled', False)
            
            if grid_enabled:
                # 6-component optimization: PV, Wind, Battery kWh, Battery kW, Genset, Grid
                bounds = [
                    (5000, 50000),    # PV capacity (kW)
                    (1, 50),          # Wind turbines (1MW each)
                    (5000, 30000),    # Battery capacity (kWh)
                    (1000, 10000),    # Battery power (kW)
                    (17000, 30000),   # Genset capacity (kW)
                    (5000, 25000)     # Grid capacity (kW)
                ]
            else:
                # 5-component optimization: PV, Wind, Battery kWh, Battery kW, Genset
                bounds = [
                    (5000, 50000),    # PV capacity (kW)
                    (1, 50),          # Wind turbines (1MW each)
                    (5000, 30000),    # Battery capacity (kWh)
                    (1000, 10000),    # Battery power (kW)
                    (17000, 30000)    # Genset capacity (kW)
                ]
            
            # Set initial conditions
            initial_conditions = [
                [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in bounds]
            ]
            
            # Run optimization
            best_result = self.system_optimizer.optimize_system(bounds, initial_conditions)
            
            if best_result:
                print(f"✓ LCOE: ${best_result['System LCOE ($/kWh)']:.4f}/kWh")
                print(f"✓ Cost: ${best_result['System NPC ($)']:,.0f}")
                print(f"✓ Demand: {best_result['Demand Met Percentage']:.1f}%")
                
                return {
                    'Latitude': latitude,
                    'Longitude': longitude,
                    'Location ID': location_id,
                    **best_result
                }
            else:
                print("✗ Optimization failed")
                return {}
                
        except Exception as e:
            print(f"✗ Error: {str(e)}")
            return {}

def main():
    """Main execution - single location example."""
    # Configuration
    yaml_file_path = "../input_yaml/input_file_chunk_0.yaml"
    api_key = "ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3"
    email = "hanrong.h99@gmail.com"
    
    # Initialize optimizer
    optimizer = HybridOptimizer(
        yaml_file_path=yaml_file_path,
        api_key=api_key,
        email=email,
        project_lifetime=25,
        discount_rate=0.0588,
        enable_flexible_load=False,
        max_load_reduction_percentage=0.2
    )
    
    # Run single location
    test_location = {
        'latitude': -33.5265,
        'longitude': 149.1588,
        'id': "TEST_LOCATION"
    }
    
    result = optimizer.process_location(
        latitude=test_location['latitude'],
        longitude=test_location['longitude'],
        location_id=test_location['id']
    )
    
    if result:
        # Save results
        results_df = pd.DataFrame([result])
        csv_filename = f"simulation_results_{test_location['latitude']}_{test_location['longitude']}.csv"
        results_df.to_csv(csv_filename, index=False)
        print(f"✓ Results saved to: {csv_filename}")
        
        print("✓ Single location optimization completed successfully!")
    else:
        print("✗ Single location optimization failed")

"""
# ========================================
# MULTIPLE LOCATION PROCESSING EXAMPLE
# ========================================
# Uncomment the code below to process multiple locations from a CSV file:

def process_multiple_locations():
    \"\"\"Process multiple locations from CSV file.\"\"\"
    # Configuration
    yaml_file_path = "../input_yaml/input_file_chunk_0.yaml"
    csv_path = "../deposit_data/auCopper_chunk_0.csv"
    output_path = "../simulation_results/simulation_results_chunk_0.csv"
    api_key = "ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3"
    email = "hanrong.h99@gmail.com"
    
    # Initialize optimizer
    optimizer = HybridOptimizer(
        yaml_file_path=yaml_file_path,
        api_key=api_key,
        email=email,
        project_lifetime=25,
        discount_rate=0.0588,
        enable_flexible_load=False,
        max_load_reduction_percentage=0.2
    )
    
    # Load location data
    try:
        location_data = pd.read_csv(csv_path)
        print(f\"Loaded {len(location_data)} locations from {csv_path}\")
    except FileNotFoundError:
        print(f\"Error: Could not find location data file: {csv_path}\")
        return
    
    # Process each location
    results = []
    for idx, row in location_data.iterrows():
        latitude = row['Latitude']
        longitude = row['Longitude']
        location_id = row.get('Location_ID', f\"LOC_{idx}\")
        
        print(f\"\\nProcessing location {idx+1}/{len(location_data)}: {location_id}\")
        
        result = optimizer.process_location(latitude, longitude, location_id)
        if result:
            results.append(result)
        else:
            print(f\"Failed to process location {location_id}\")
    
    # Save all results
    if results:
        results_df = pd.DataFrame(results)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        results_df.to_csv(output_path, index=False)
        print(f\"\\n✓ All results saved to: {output_path}\")
        print(f\"✓ Successfully processed {len(results)}/{len(location_data)} locations\")
        
        # Display summary statistics
        print(f\"\\nSummary Statistics:\")
        print(f\"  Average LCOE: ${results_df['System LCOE ($/kWh)'].mean():.4f}/kWh\")
        print(f\"  LCOE Range: ${results_df['System LCOE ($/kWh)'].min():.4f} - ${results_df['System LCOE ($/kWh)'].max():.4f}/kWh\")
        print(f\"  Average Demand Met: {results_df['Demand Met Percentage'].mean():.1f}%\")
    else:
        print(\"\\n✗ No successful optimizations completed\")

# To run multiple location processing, uncomment the line below:
# process_multiple_locations()
"""

if __name__ == "__main__":
    main()